## 实验3：常量
接下来写：

In [1]:
#include <iostream>

In [2]:
{
    int value = 10;
    const int& ref = value;

    std::cout << "value = " << value << '\n';
    std::cout << "ref   = " << ref << '\n';

    value = 20;

    std::cout << "value = " << value << '\n';
    std::cout << "ref   = " << ref << '\n';
}

value = 10
ref   = 10
value = 20
ref   = 20


这里的`const int& ref = value;`
意味着：`ref`是`value`的**只读引用**

不是说`value`变成`const`，而是：
`value`依然可以：
```C++
value = 20;
```

但是`ref`:
```C++
ref = 20;
```
不被允许

### `const T&`是 C++ API 里极其重要的模式
比如：
```C++
void print(const User& user);
```
它告诉调用者三个信息：
1. User 一定存在
2. 不复制 User 对象
3. 函数不会通过这个引用修改 User

也就是：**只读引用**


之后看到：
```C++
const std::string& name;
```
首先想到的应该是：
1. 这个函数值借用 `std::string`
2. 没有所有权（ownership）
3. 也不会修改这个值

---


### 为什么不直接使用值传递？
例如：
- 值传递：
```C++
void print(std::string value);
```
意味着可能发生：
```
original string

      │
      │ copy
      ▼

new string
```

- 引用传递：
```C++
void print(const std::string& value);
```
则是：
```
original string
      ▲
      │ borrow
      │
    value
```
没有为了调用函数而复制一个 string 值，仅借用


所以经典的 C++ API：
```C++
void process(const std::vector<int>& data);
```
原因就在这里

---


### 重点：常量与指针（极易混淆）
编写新的实验代码：

In [6]:
{
    int a = 10;
    int b = 20;

    const int* pointer_to_const = &a;

    int* const const_pointer = &a;

    const int* const const_pointer_to_const = &a;

    std::cout << *pointer_to_const << '\n';
    std::cout << *const_pointer << '\n';
    std::cout << *const_pointer_to_const << '\n';
}

10
10
10


#### 必须分清三个类型：
1. 常量指针
```C++
const int* ptr;
// or
int const* ptr;
```
意思是：这个指针指向一个常量
```
pointer
   │
   ▼
const int
```
指针本身可以被修改：
```C++
ptr = &b;
```
但是指向的内容不可以被修改：
```C++
*ptr = 100;
```
这是不被允许的

2. 指针常量
```C++
int* const ptr = &a;
```
这表示：指针本身是一个常量
```
const pointer
      │
      ▼
     int
```
不可以修改地址：
```C++
ptr = &b; // 不被允许
```

但是可以修改指向对象的值：
```C++
*ptr = 100; // 合法
```

3. 常量指针常量
```C++
const int* const ptr = &a;
```

这表示：指针本身和指向的对象都是常量
```
const pointer
      │
      ▼
const int
```
地址和对象都不可以被修改：
```C++
ptr = &b;   // error
*ptr = 100; // error
```


#### 实用的辨别方法
看到：
```C++
const int* ptr;
```
从`ptr`往左读：
```
ptr
指向（*）
const int
```


看到：
```C++
int* const ptr
```
同样的方式：
```
ptr
是一个常量
指向（*）
int
```


看到：
```C++
const int* const ptr;
```

依旧：
```
ptr
是一个常量
指向（*）
const int
```

这在 C ABI 中很常见：
```C
int sdk_process(
    const uint8_t* input,
    size_t length
);
```
这表示：`SDK 借用输入 buffer，而不修改它`
所以`const`不只是**编译器限制**，也是**API 规范**

---
